# 🚀 Building Your First AI Agent with Agent Framework - Step-by-Step Tutorial

## 👋 Welcome, Developer!

This tutorial will guide you through creating your first AI agent from scratch using the **Microsoft Agent Framework**. By the end, you'll have a working travel planning agent that can use tools, manage conversations, and provide intelligent responses.

**What You'll Learn:**
- ✅ How to set up and configure an AI agent
- ✅ How to create and register custom tools
- ✅ How to manage conversation state and threads
- ✅ How to integrate with Azure AI Foundry (Azure OpenAI)
- ✅ How agents make decisions and use tools

**Prerequisites:**
- Basic Python knowledge
- Azure OpenAI access (or Azure AI Foundry project)
- 15-20 minutes of your time

Let's build something amazing! 🎯

## 📚 Step 1: Understanding the Agent Framework Architecture

Before we code, let's understand what we're building:

```
┌─────────────────────────────────────────────────────────┐
│                    AI AGENT                              │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐  │
│  │   LLM Model  │  │    Tools     │  │    Memory    │  │
│  │   (Brain)    │  │ (Abilities)  │  │ (Conversation)│ │
│  └──────────────┘  └──────────────┘  └──────────────┘  │
└─────────────────────────────────────────────────────────┘
         ↓                    ↓                  ↓
    Reasoning          Tool Execution      Context Tracking
```

**Key Components:**

1. **ChatAgent**: The main orchestrator that coordinates everything
2. **Chat Client**: Connects to the LLM (Azure OpenAI in our case)
3. **Tools**: Functions the agent can call to perform actions
4. **Thread**: Manages conversation history and state
5. **Instructions**: The agent's "personality" and behavior guidelines

**How It Works:**
1. User sends a message
2. Agent analyzes the request using the LLM
3. Agent decides if it needs to use tools
4. Agent executes tools if needed
5. Agent formulates a response
6. Response is sent back to user

Now let's build this!

## 📦 Step 2: Install Required Packages

First, we need to install the Agent Framework. According to our requirements.txt, we're using:
- `agent-framework-core==1.0.0b251016`
- `python-dotenv` for environment variables
- `azure-identity` for Azure authentication

**What each package does:**
- **agent-framework-core**: The main framework for building agents
- **python-dotenv**: Loads configuration from .env files (keeps secrets safe)
- **azure-identity**: Handles Azure authentication (API keys or managed identity)

In [ ]:
# Install the required packages
# Note: This uses the exact versions from requirements.txt
! pip install agent-framework-core==1.0.0b251016 python-dotenv==1.0.1 azure-identity==1.19.0 -q

## 🔧 Step 3: Import Required Libraries

Let's import everything we need and understand what each import does:

**Standard Library:**
- `os`: Access environment variables
- `randint`: Generate random numbers (for our destination picker)

**Third-party:**
- `load_dotenv`: Load configuration from .env file

**Agent Framework:**
- `ChatAgent`: Main agent class
- `AzureAIAgentClient`: Client for Azure AI Foundry/OpenAI
- `AzureCliCredential`: Azure authentication using CLI credentials

In [ ]:
# Standard library imports
import os
from random import randint

# Environment configuration
from dotenv import load_dotenv

# Agent Framework components
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient

# Load environment variables from .env file
load_dotenv()

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 🛠️ Step 4: Create a Custom Tool

Tools are functions that give your agent **abilities**. Think of them as skills the agent can use.

**In this example, we'll create a tool that:**
- Selects a random vacation destination
- Returns the destination name as a string

**Important Tool Design Principles:**
1. **Clear Name**: Use descriptive function names (the agent sees these)
2. **Type Hints**: Always specify input/output types
3. **Docstring**: Explain what the tool does (the agent reads this!)
4. **Simple Logic**: Tools should do one thing well
5. **Return Values**: Always return data the agent can use

**How the Agent Uses Tools:**
1. User: "Plan me a trip"
2. Agent thinks: "I need a destination. I have a tool called 'get_random_destination'"
3. Agent calls the tool
4. Tool returns: "Paris, France"
5. Agent uses this to create a travel plan

In [3]:
def get_random_destination() -> str:
    """
    Get a random vacation destination from a curated list.
    
    This tool selects one destination randomly from 10 popular 
    travel locations around the world.
    
    Returns:
        str: The name of a vacation destination (e.g., "Paris, France")
    
    Example:
        >>> destination = get_random_destination()
        >>> print(destination)
        'Tokyo, Japan'
    """
    # Our curated list of destinations
    destinations = [
        "Barcelona, Spain",
        "Paris, France",
        "Berlin, Germany",
        "Tokyo, Japan",
        "Sydney, Australia",
        "New York, USA",
        "Cairo, Egypt",
        "Cape Town, South Africa",
        "Rio de Janeiro, Brazil",
        "Bali, Indonesia"
    ]
    
    # Select a random destination
    random_index = randint(0, len(destinations) - 1)
    selected_destination = destinations[random_index]
    
    return selected_destination

# Test the tool
test_destination = get_random_destination()
print(f"🎲 Random destination: {test_destination}")
print("✅ Tool created successfully!")

🎲 Random destination: Bali, Indonesia
✅ Tool created successfully!


## 🤖 Step 5: Configure the Azure AI Client

The **Chat Client** is how your agent communicates with the AI model (the "brain").

**What We're Configuring:**

1. **Authentication**: Using Azure CLI credentials (you must run `az login` first)
2. **Model Deployment**: The specific AI model to use (e.g., gpt-4o)
3. **Endpoint**: Your Azure AI Foundry project URL

**Configuration Options:**

**Authentication Methods:**
- `AzureCliCredential()`: Uses your Azure CLI login (good for dev)
- API Key: Set `api_key="your-key"` instead (good for production)
- Managed Identity: For production deployments in Azure

**Model Deployment Names:**
- `gpt-4o`: Most capable, slower, more expensive
- `gpt-4o-mini`: Fast, cost-effective, great for most tasks
- `gpt-4`: Previous generation, still very capable

**Important:** Make sure you've run `az login` before executing this cell!

In [6]:
# Configure the Azure AI Agent Client
# This connects your agent to the AI model

# Get credentials from environment variables
api_key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")
foundry_openai_endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
model_name = os.getenv("AZURE_AI_FOUNDRY_MODEL", "gpt-4o")

if not api_key:
    raise ValueError(
        "AZURE_AI_FOUNDRY_API_KEY environment variable is required. "
        "Please set it in your .env file or environment."
    )


## 📝 Step 6: Define Agent Instructions

**Instructions** are like the agent's job description and personality guide. They tell the agent:
- What its role is
- How to behave
- What tasks it can perform
- How to interact with users

**Best Practices for Writing Instructions:**

1. **Be Specific**: Clearly define the agent's role and capabilities
2. **Set Boundaries**: Explain what the agent should and shouldn't do
3. **Define Behavior**: Specify tone, style, and interaction patterns
4. **Provide Examples**: Show the agent how to respond
5. **Handle Edge Cases**: Explain what to do in special situations

**Instruction Components:**
- **Role**: "You are a helpful travel planning assistant"
- **Capabilities**: "You can suggest destinations and plan itineraries"
- **Behavior**: "Always be friendly and enthusiastic"
- **Constraints**: "Only suggest destinations when asked"

In [5]:
# Agent name (for logging and identification)
AGENT_NAME = "TravelPlannerAgent"

# Agent instructions (the agent's "personality" and behavior guide)
AGENT_INSTRUCTIONS = """
You are a friendly and helpful AI Travel Planning Assistant.

YOUR ROLE:
- Help users plan exciting and memorable vacations
- Suggest destinations and create detailed itineraries
- Provide practical travel advice and tips

YOUR CAPABILITIES:
You have access to a tool called 'get_random_destination' that can:
- Select a random vacation destination from popular global locations
- Use this tool when users ask for destination suggestions

HOW TO BEHAVE:
1. **Be Enthusiastic**: Show excitement about travel and destinations
2. **Be Helpful**: Provide detailed, actionable information
3. **Be Clear**: Structure your responses in an easy-to-read format
4. **Be Respectful**: Honor user preferences and constraints

WHEN PLANNING A TRIP:
1. If the user hasn't specified a destination, use get_random_destination
2. If the user specified a destination, plan for that location
3. Include these elements in your itinerary:
   - Morning activities
   - Afternoon activities
   - Evening recommendations
   - Local cuisine suggestions
   - Practical tips (transport, budget, best time to visit)

EXAMPLE INTERACTION:
User: "Plan me a day trip"
You: 
1. Call get_random_destination to get a location
2. Create a detailed day plan for that destination
3. Present it in a friendly, structured format

Remember: You're not just providing information - you're helping people create 
amazing travel memories!
"""

print(f"✅ Agent instructions defined!")
print(f"📋 Agent Name: {AGENT_NAME}")
print(f"📄 Instructions length: {len(AGENT_INSTRUCTIONS)} characters")

✅ Agent instructions defined!
📋 Agent Name: TravelPlannerAgent
📄 Instructions length: 1397 characters


## 🎯 Step 7: Create the Agent

Now we bring everything together to create our agent!

**The ChatAgent Constructor:**

```python
ChatAgent(
    name="...",              # Agent identifier
    chat_client=...,         # Connection to AI model
    instructions="...",      # Behavior guidelines
    tools=[...]              # Available functions
)
```

**What Happens During Creation:**
1. Agent registers the chat client for model access
2. Agent loads and validates instructions
3. Agent registers all tools and their metadata
4. Agent prepares internal state management
5. Agent is ready to handle requests

**Key Parameters Explained:**
- `name`: Identifies this agent (useful when you have multiple agents)
- `chat_client`: The AI model connection we configured earlier
- `instructions`: The behavior guide we wrote
- `tools`: List of functions the agent can call

After this step, your agent is fully initialized and ready to work!

In [11]:
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import DefaultAzureCredential
# Azure version using OpenAI-compatible client

chat_client= AzureOpenAIChatClient(
    endpoint=foundry_openai_endpoint,  # Azure OpenAI endpoint
    api_key=api_key,
    deployment_name=model_name  # Your deployment name
)


In [12]:
# Import ChatAgent (missing from earlier imports)
from agent_framework import ChatAgent

# Create the ChatAgent by combining all components
agent = ChatAgent(
    # Identifier for this agent
    name=AGENT_NAME,
    
    # The AI model client we configured
    chat_client=chat_client,
    
    # The behavior instructions we wrote
    instructions=AGENT_INSTRUCTIONS,
    
    # The tools available to this agent
    tools=[get_random_destination]
)

print("✅ Agent created successfully!")
print(f"🤖 Agent Name: {agent.name}")
print(f"📱 Status: Ready to chat!")

✅ Agent created successfully!
🤖 Agent Name: TravelPlannerAgent
📱 Status: Ready to chat!


## 💬 Step 8: Create a Conversation Thread

**Threads** manage conversation state and history. Think of them as individual chat sessions.

**Why Do We Need Threads?**
- **Memory**: Keeps track of what was said in the conversation
- **Context**: Allows the agent to reference previous messages
- **Isolation**: Separates different conversations
- **State**: Maintains conversation-specific data

**Thread Lifecycle:**
```
Create Thread → Send Messages → Agent Responds → Continue Conversation
     ↓              ↓                ↓                    ↓
  [New ID]      [Added]         [Added]              [Context]
```

**Use Cases:**
- **Single Thread**: One ongoing conversation
- **Multiple Threads**: Different users or conversation topics
- **Thread per Session**: Each user session gets its own thread

**Important:** The thread stores the entire conversation history, so the agent can:
- Remember what you asked earlier
- Reference previous responses
- Maintain context across multiple turns

In [13]:
# Create a new conversation thread
# This is like starting a new chat session
thread = agent.get_new_thread()

print("✅ Conversation thread created!")
print(f"🧵 Thread ID: {thread.id if hasattr(thread, 'id') else 'generated'}")
print(f"📊 Messages in thread: 0 (new conversation)")
print("\n💡 Pro Tip: You can create multiple threads for different conversations!")

✅ Conversation thread created!
🧵 Thread ID: generated
📊 Messages in thread: 0 (new conversation)

💡 Pro Tip: You can create multiple threads for different conversations!


## 🚀 Step 9: Run Your First Agent Request

This is where the magic happens! Let's send a message to our agent.

**What Happens Behind the Scenes:**

1. **Message Received**: Your request enters the agent system
2. **Context Building**: Agent loads instructions + conversation history
3. **LLM Analysis**: AI model analyzes the request
4. **Decision Making**: Agent decides if it needs to use tools
5. **Tool Execution**: If needed, agent calls get_random_destination
6. **Response Generation**: Agent formulates the final response
7. **Return**: Response is sent back to you

**The Agent's Thought Process:**
```
User: "Plan me a day trip"
Agent thinks: "I need a destination. I have a tool for that!"
Agent calls: get_random_destination()
Tool returns: "Tokyo, Japan"
Agent thinks: "Now I'll create a day plan for Tokyo"
Agent responds: [Detailed Tokyo itinerary]
```

**The `await` keyword**: 
- Agent Framework uses async/await for efficient I/O
- `await` means "wait for this to complete before continuing"
- This allows multiple requests to be handled efficiently

In [14]:
# Send a message to the agent and wait for response
print("🎬 Sending request to agent...\n")

# The user's request
user_message = "Plan me a day trip"
print(f"👤 You: {user_message}")
print("\n⏳ Agent is thinking...")


# Run the agent (this is async, so we use await)
response = await agent.run(user_message, thread=thread)

print("\n✅ Response received!")
print(f"📨 Total messages in conversation: {len(response.messages)}")


🎬 Sending request to agent...

👤 You: Plan me a day trip

⏳ Agent is thinking...

✅ Response received!
📨 Total messages in conversation: 3

✅ Response received!
📨 Total messages in conversation: 3


## 📖 Step 10: Extract and Display the Response

The agent returns a response object that contains:
- **messages**: All messages in the conversation (including tool calls)
- **metadata**: Information about the execution
- **status**: Success/failure indicators

**Understanding the Message Structure:**

```python
response.messages = [
    UserMessage,           # Your original message
    AssistantMessage,      # Agent's first thought
    ToolCallMessage,       # Tool execution request
    ToolResultMessage,     # Tool's return value
    AssistantMessage       # Final response to user
]
```

**We want the last message** because that's the agent's final response to the user.

**Message Content Structure:**
- `message.contents`: List of content parts
- `contents[0].text`: The actual text response

**Why this structure?** It allows messages to contain multiple types of content (text, images, files, etc.)

In [39]:
# Extract the agent's final response
# The last message is the agent's final answer
last_message = response.messages[-1]

# Get the text content from the message
agent_response_text = last_message.contents[0].text

# Display the response
print("="*60)
print("🤖 AGENT RESPONSE")
print("="*60)
print(agent_response_text)
print("="*60)

🤖 AGENT RESPONSE
How exciting! We're planning a day trip to **Bali, Indonesia**, one of the most beautiful island destinations on the planet. Bali is famous for its stunning beaches, lush greenery, mesmerizing temples, and lively culture. Here’s your fabulous day itinerary:

---

### **Morning: Seaside Adventure**
- **Start your day at**: Sanur Beach
  - Begin with a sunrise walk on the peaceful sandy shore of Sanur Beach. The early morning sky in Bali is breathtaking.
  - Rent a bicycle to explore the beach promenade or simply relax with your toes in the sand.

- **Activity**: Try Stand-Up Paddleboarding
  - Sanur is a great spot for beginners to try paddleboarding due to its calm waters.

- **Breakfast**: Stop at **Soul in a Bowl**
  - Savor a delicious breakfast with smoothie bowls, avocado on toast, and freshly brewed Balinese coffee.

---

### **Afternoon: Culture & Exploration**
- **Visit**: Tanah Lot Temple
  - One of Bali’s iconic sea temples situated on a rock formation by the

## 🔄 Step 11: Continue the Conversation

One of the most powerful features of agents is **conversation continuity**. The agent remembers what was discussed!

**How Conversation Memory Works:**

1. **Thread Stores History**: All messages are saved in the thread
2. **Context Loading**: Each new request includes previous messages
3. **Contextual Understanding**: Agent can reference earlier parts of the conversation

**Why This Is Powerful:**
- User: "I don't like that destination"
- Agent knows: "That destination" refers to the previous suggestion
- Agent can: Suggest a different destination intelligently

**Without Memory:**
```
User: "Plan a trip to Paris"
Agent: [Paris itinerary]
User: "What about the weather?"
Agent: "Weather where?" ❌
```

**With Memory:**
```
User: "Plan a trip to Paris"
Agent: [Paris itinerary]
User: "What about the weather?"
Agent: "In Paris, the weather..." ✅
```

In [41]:
# Send a follow-up message that references the previous conversation
print("\n🔄 Continuing the conversation...\n")

# Follow-up message
follow_up_message = "I don't like that destination. Can you suggest another vacation spot and plan a day there?"
print(f"👤 You: {follow_up_message}")
print("\n⏳ Agent is thinking...")

# Run the agent again with the same thread (maintains conversation history)
response2 = await agent_azure.run(follow_up_message, thread=thread)

# Extract and display the new response
last_message2 = response2.messages[-1]
agent_response_text2 = last_message2.contents[0].text

print("\n" + "="*60)
print("🤖 AGENT RESPONSE")
print("="*60)
print(agent_response_text2)
print("="*60)

print(f"\n📊 Conversation now has {len(response2.messages)} messages")


🔄 Continuing the conversation...

👤 You: I don't like that destination. Can you suggest another vacation spot and plan a day there?

⏳ Agent is thinking...

🤖 AGENT RESPONSE
How about Berlin, Germany? It's a vibrant city full of history, culture, and great food! Let me plan an exciting day for you in Berlin.

---

### **A Memorable Day in Berlin**
**Morning: Exploring History**
- Start your day with a visit to the iconic **Brandenburg Gate**, an 18th-century neoclassical monument and symbol of peace.
- Walk over to **Reichstag Building**, the German Parliament, and enjoy breathtaking views of Berlin from the glass dome (advance registration recommended).
- Head to the **Holocaust Memorial**, a haunting yet beautiful tribute to the victims of the Holocaust.

**Afternoon: Art and Culture**
- Have lunch at **Markthalle Neun**, a buzzing food market where you can savor German dishes like currywurst or schnitzel and enjoy international cuisines.
- Spend the afternoon on **Museum Island** —

## 🎓 Step 12: Understanding What Just Happened

Let's break down the agent's behavior:

### First Request: "Plan me a day trip"

**Agent's Process:**
1. ✅ Received user request
2. ✅ Identified need for a destination
3. ✅ Called `get_random_destination()` tool
4. ✅ Received random destination (e.g., "Tokyo, Japan")
5. ✅ Generated detailed day itinerary
6. ✅ Formatted response for user

### Second Request: "I don't like that destination..."

**Agent's Process:**
1. ✅ Loaded conversation history
2. ✅ Understood "that destination" = previous suggestion
3. ✅ Called `get_random_destination()` again for new location
4. ✅ Ensured different destination
5. ✅ Generated new itinerary
6. ✅ Maintained conversational flow

### Key Observations:

**🧠 Intelligence:**
- Agent understood context without explicit reminders
- Made appropriate tool choices automatically
- Maintained conversation coherence

**🛠️ Tool Usage:**
- Agent decided when to call tools
- Used tool results to enhance responses
- Could call multiple tools if needed

**💬 Conversation:**
- Remembered previous messages
- Built upon earlier context
- Maintained natural flow

## 🎯 Congratulations! You've Built Your First AI Agent!

### What You've Learned:

✅ **Agent Architecture**: Understanding how agents work
✅ **Tool Creation**: Building custom capabilities
✅ **Azure Integration**: Connecting to AI models
✅ **Instructions**: Defining agent behavior
✅ **Conversation Management**: Handling stateful interactions
✅ **Agent Framework API**: Using the core components

### 🚀 Next Steps - Try These Challenges:

#### Challenge 1: Add More Tools
```python
def get_weather_info(city: str) -> str:
    """Get weather information for a city (mock data)"""
    return f"Weather in {city}: Sunny, 75°F"
```

#### Challenge 2: Enhance Instructions
- Add budget-conscious travel tips
- Include accessibility information
- Support multiple languages

#### Challenge 3: Add User Preferences
```python
def save_user_preference(preference: str) -> str:
    """Remember user's travel preferences"""
    # Save to file or database
    return f"Saved preference: {preference}"
```

#### Challenge 4: Multi-turn Planning
Create a multi-day itinerary with different tools for each day

#### Challenge 5: Add Validation
```python
def validate_destination(destination: str) -> bool:
    """Check if destination is safe/accessible"""
    # Add validation logic
    return True
```

### 📚 Additional Resources:

- **Agent Framework Docs**: [Learn more about advanced features](https://github.com/microsoft/agent-framework)
- **Azure AI Foundry**: [Explore model capabilities](https://learn.microsoft.com/azure/ai-studio/)
- **Tool Design Patterns**: [Best practices for tool creation](https://learn.microsoft.com/azure/ai-studio/concepts/tools)

### 💡 Key Takeaways:

1. **Agents = Intelligence + Tools + Memory**
2. **Tools give agents superpowers**
3. **Instructions shape agent behavior**
4. **Threads maintain conversation context**
5. **The framework handles the complexity**

### 🎉 You're now ready to build production AI agents!

Keep experimenting, keep building, and most importantly - have fun! 🚀